# Sampling VWW dataset

In [ ]:
import os
import random
from PIL import Image
import matplotlib.pyplot as plt

dataset_dir = "VWW/vww-s256/val/"
class_0_dir = os.path.join(dataset_dir, "0")
class_1_dir = os.path.join(dataset_dir, "1")

images_0 = [
    os.path.join(class_0_dir, img)
    for img in os.listdir(class_0_dir)
    if img.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
]

images_1 = [
    os.path.join(class_1_dir, img)
    for img in os.listdir(class_1_dir)
    if img.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
]

random.shuffle(images_0)
random.shuffle(images_1)

selected_0 = images_0[:8]
selected_1 = images_1[:8]

selected_images = []
for img1, img0 in zip(selected_1, selected_0):
    selected_images.append((img1, 1))
    selected_images.append((img0, 0))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for ax, (img_path, label) in zip(axes.flatten(), selected_images):
    img = Image.open(img_path)
    img = img.resize((96,96))
    ax.imshow(img)

    if label == 1:
      title = "Person"
    else:
      title = "Not a Person"
      
    ax.set_title(f"{title}", fontsize=14)
    ax.axis("off")

plt.tight_layout()

output_file = "article-images/dataset_grid.pdf"
plt.savefig(output_file, format="pdf", dpi=300, bbox_inches="tight")

plt.show()

print(f"Figura salva em: {output_file}")

# Ploting VWW-C directory

In [ ]:
import matplotlib.pyplot as plt

LINES = [
    "VWW-C/",
    "├── Brightness/",
    "│   ├── 1/",
    "│   │   ├── 0/",
    "│   │   │   ├── COCO_val2014_000000000042.jpg",
    "│   │   │   ├── ...",
    "│   │   │   └── COCO_val2014_000000000081.jpg",
    "│   │   └── 1/",
    "│   │       └── ...",
    "│   ├── 2/",
    "│   │   └── ...",
    "│   ├── ...",
    "│   └── 5/",
    "│       └── ...",
    "├── ...",
    "└── Zoom Blur/",
    "    └── ...",
]

FONTSIZE = 9
LINE_H   = FONTSIZE * 1.6 
DPI      = 300

n_lines  = len(LINES)
max_len  = max(len(l) for l in LINES)

char_w_pt = FONTSIZE * 0.60
fig_w_in  = (max_len * char_w_pt + 40) / 72
fig_h_in  = (n_lines * LINE_H + 40) / 72

fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.axis("off")

for i, line in enumerate(LINES):
    ax.text(
        0.01, 1.0 - (i / n_lines),
        line,
        transform=ax.transAxes,
        fontsize=FONTSIZE,
        fontfamily="monospace",
        va="top",
        color="#111111",
    )

plt.tight_layout(pad=0.3)
out = "article-images/dataset_structure.pdf"
fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close()
print(f"Salvo: {out}")

# Sampling VWW-C dataset

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

DATASET_CLEAN = "VWW/vww-s256/val/0"
DATASET_CORR  = "VWW-C"
SEVERITY      = "5"
CLASS         = "0"

CORRUPTIONS = [
    "Brightness", "Contrast", "Defocus Blur", "Elastic",
    "Fog", "Frost", "Gaussian Noise", "Glass Blur",
    "Impulse Noise", "JPEG", "Motion Blur", "Pixelate",
    "Shot Noise", "Snow", "Zoom Blur"
]

def first_image(folder):
    exts = (".jpg", ".jpeg", ".png")
    for f in sorted(os.listdir(folder)):
        if f.lower().endswith(exts):
            return os.path.join(folder, f)
    raise FileNotFoundError(f"Nenhuma imagem encontrada em: {folder}")

images = []

images.append(("Clean", first_image(DATASET_CLEAN)))

for corr in CORRUPTIONS:
    folder = os.path.join(DATASET_CORR, corr, SEVERITY, CLASS)
    images.append((corr, first_image(folder)))

ROWS, COLS = 4, 4
fig, axes = plt.subplots(ROWS, COLS, figsize=(12, 12))
fig.patch.set_facecolor("white")

for idx, ax in enumerate(axes.flat):
    title, path = images[idx]
    img = mpimg.imread(path)
    ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
    ax.set_title(title, fontsize=18, fontweight="bold" if title == "Clean" else "normal",
                 pad=4)
    ax.axis("off")
    
plt.tight_layout(pad=0.5)

out = "article-images/corruption_grid.pdf"
fig.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print(f"Salvo: {out}")

# Fitness Robust Analysis

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

CONFIGS = [
    ("18M 128KB", 18, 128),
    ("18M 256KB", 18, 256),
    ("18M 512KB", 18, 512),
    ("52M 128KB", 52, 128),
    ("52M 256KB", 52, 256),
    ("52M 512KB", 52, 512),
]
DATA_TRAD = "BRACIS-results/data/standard"

ABBREV = {
    "Clean":         "Clean",
    "Brightness":    "Brt",
    "Contrast":      "Cnt",
    "Defocus Blur":  "DfB",
    "Elastic":       "Ela",
    "Fog":           "Fog",
    "Frost":         "Frs",
    "Gaussian Noise":"GsN",
    "Glass Blur":    "GlB",
    "Impulse Noise": "ImN",
    "JPEG":          "JPG",
    "Motion Blur":   "MtB",
    "Pixelate":      "Pxl",
    "Shot Noise":    "ShN",
    "Snow":          "Snw",
    "Zoom Blur":     "ZmB",
}

def load_config(macs, mem):
    clean_path = f"{DATA_TRAD}/cfg_{macs}_{mem}_acc_list.json"
    corr_path  = f"{DATA_TRAD}/cfg_{macs}_{mem}_corruption_results.json"

    with open(clean_path) as f:
        clean_accs = np.array(json.load(f))
    with open(corr_path) as f:
        results = json.load(f)

    samples     = list(results.keys())
    corruptions = list(results[samples[0]]["corruptions"].keys())
    N, C        = len(samples), len(corruptions)

    acc = np.zeros((N, C))
    for i, sample in enumerate(samples):
        for j, corr in enumerate(corruptions):
            acc[i, j] = results[sample]["corruptions"][corr]["5"]

    acc_full   = np.hstack([clean_accs[:, None], acc])
    labels_all = ["Clean"] + corruptions
    return acc_full, labels_all

all_data   = []
labels_ref = None

for cfg_name, macs, mem in CONFIGS:
    acc_full, labels_all = load_config(macs, mem)
    all_data.append((cfg_name, acc_full))
    if labels_ref is None:
        labels_ref = labels_all

acc_first  = all_data[0][1]

corr_medians = np.median(acc_first[:, 1:], axis=0)
corr_order   = np.argsort(corr_medians)[::-1]          
final_order  = np.concatenate(([0], corr_order + 1))   

labels_ordered = [labels_ref[i] for i in final_order]
abbrev_ordered = [ABBREV.get(l, l) for l in labels_ordered]

plt.rcParams.update({
    "font.family":      "monospace",
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "text.color":       "#111111",
    "axes.labelcolor":  "#111111",
    "xtick.color":      "#333333",
    "ytick.color":      "#333333",
    "axes.edgecolor":   "#AAAAAA",
    "grid.color":       "#DDDDDD",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

BP_STYLE = dict(
    patch_artist=True,
    boxprops=dict(facecolor="#a8c8eb", color="#2a6db5"),
    medianprops=dict(color="#e07b00", linewidth=2),
    whiskerprops=dict(color="#2a6db5", linewidth=1.2),
    capprops=dict(color="#2a6db5", linewidth=1.2),
    flierprops=dict(marker="o", color="#cc2200", markerfacecolor="#cc2200",
                    markersize=4, alpha=0.7),
)

N_CONFIGS = len(CONFIGS)
N_BOXES   = len(final_order)   # 16

fig, axes = plt.subplots(
    N_CONFIGS, 1,
    figsize=(16, 3.2 * N_CONFIGS),
    sharex=True
)
fig.patch.set_facecolor("white")

for ax, (cfg_name, acc_full) in zip(axes, all_data):
    data_ordered = [acc_full[:, i] for i in final_order]
    global_mean  = acc_full[:, 1:].mean()

    ax.boxplot(data_ordered, **BP_STYLE)
    ax.axhline(global_mean, linestyle=":", color="#e07b00", linewidth=1.4,
               label=f"MCA = {global_mean:.2f}%", zorder=2)

    ax.set_ylim(50, 94)
    ax.tick_params(axis="y", labelsize=18)
    ax.set_ylabel("Accuracy (%)", fontsize=18)
    ax.set_title(cfg_name, fontsize=16, loc="center", pad=4)
    ax.legend(fontsize=16, loc="upper right")
    ax.grid(axis="y", zorder=0, alpha=0.5)

axes[-1].set_xticks(range(1, N_BOXES + 1))
axes[-1].set_xticklabels(abbrev_ordered, fontsize=20)

fig.suptitle(
    "MCA boxplots for baseline sub-networks, organized by hardware configuration",
    fontsize=24, y=1.01
)

plt.tight_layout()
out = "article-images/mca_boxplots_baseline.pdf"
fig.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print(f"Salvo: {out}")

# Generating CSV Files


In [ ]:
import numpy as np
import json
import pandas as pd

corrupted_accs_filename = "path/to/corrupted_accuracies.json"
label = "Label"

with open(corrupted_accs_filename) as f:
    results = json.load(f)

samples     = list(results.keys())
N           = len(samples)
corruptions = list(results[samples[0]]["corruptions"].keys())
C           = len(corruptions)

corruption_data = {corruption: [] for corruption in corruptions}

sample_means = []

for sample in samples:
    sample_accs = []
    
    for corruption in corruptions:
        severity_key = list(results[sample]["corruptions"][corruption].keys())[0]
        acc = results[sample]["corruptions"][corruption][severity_key]
        
        corruption_data[corruption].append(acc)
        sample_accs.append(acc)
    
    sample_means.append(np.mean(sample_accs))

corruption_stats = []
for corruption in corruptions:
    accs = np.array(corruption_data[corruption])
    corruption_stats.append({
        'Corruption': corruption,
        'Mean_Accuracy': np.mean(accs),
        'Std_Deviation': np.std(accs, ddof=1)
    })

corruption_stats_file = f'tabelas/corruption_statistics_{label}.csv'
df_corruptions = pd.DataFrame(corruption_stats)
df_corruptions = df_corruptions.sort_values('Mean_Accuracy', ascending=False)
df_corruptions.to_csv(corruption_stats_file, index=False, float_format='%.4f')

global_stats = []
for i, sample in enumerate(samples):
    global_stats.append({
        'Sample': sample,
        'Mean_Accuracy': sample_means[i]
    })

df_global = pd.DataFrame(global_stats)

overall_mean = np.mean(sample_means)
overall_std = np.std(sample_means, ddof=1)

summary_row = pd.DataFrame([{
    'Sample': 'OVERALL',
    'Mean_Accuracy': overall_mean
}])

df_global['Std_Deviation'] = ''
summary_row['Std_Deviation'] = overall_std

global_file = f'tabelas/global_statistics_{label}.csv'
df_global = pd.concat([df_global, summary_row], ignore_index=True)
df_global.to_csv(global_file, index=False, float_format='%.4f')

print(f"✓ Processadas {N} amostras e {C} corrupções")
print(f"✓ Tabela por corrupção salva em: {corruption_stats_file}")
print(f"✓ Tabela global salva em: {global_file}")
print(f"\nResumo Global:")
print(f"  Acurácia média: {overall_mean:.4f} ± {overall_std:.4f}")

# Per Corruption Evaluation

In [ ]:
import json
import numpy as np
import pandas as pd
import os

CONFIGS_18M = [
    ("18M_128KB",        18, 128, False),
    ("18M_128KB_Robust", 18, 128, True),
    ("18M_256KB",        18, 256, False),
    ("18M_256KB_Robust", 18, 256, True),
    ("18M_512KB",        18, 512, False),
    ("18M_512KB_Robust", 18, 512, True),
]

CONFIGS_52M = [
    ("52M_128KB",        52, 128, False),
    ("52M_128KB_Robust", 52, 128, True),
    ("52M_256KB",        52, 256, False),
    ("52M_256KB_Robust", 52, 256, True),
    ("52M_512KB",        52, 512, False),
    ("52M_512KB_Robust", 52, 512, True),
]

DATA_TRAD   = "BRACIS-results/standard-method"
DATA_ROBUST = "BRACIS-results/fitness-robust"
OUT_DIR     = "tables"
os.makedirs(OUT_DIR, exist_ok=True)

def load_config(macs, mem, robust=False):
    if not robust:
        clean_path = f"{DATA_TRAD}/cfg_{macs}_{mem}_acc_list.json"
        corr_path  = f"{DATA_TRAD}/cfg_{macs}_{mem}_corruption_results.json"
    else:
        clean_path = f"{DATA_ROBUST}/ACCS_{macs}_{mem}_clean_ROBUST.json"
        corr_path  = f"{DATA_ROBUST}/ACCS_{macs}_{mem}_corrupted_ROBUST.json"

    with open(clean_path) as f:
        clean_accs = np.array(json.load(f))

    with open(corr_path) as f:
        results = json.load(f)

    samples     = list(results.keys())
    corruptions = list(results[samples[0]]["corruptions"].keys())
    N, C        = len(samples), len(corruptions)

    acc = np.zeros((N, C))
    for i, sample in enumerate(samples):
        for j, corr in enumerate(corruptions):
            acc[i, j] = results[sample]["corruptions"][corr]["5"]

    return clean_accs, acc, corruptions

def compute_diff(clean_accs, acc):
    """
    Acc diff = clean_mean - corr_mean  (por corrupção)
    Retorna array de shape (C,)
    """
    clean_mean = clean_accs.mean()
    corr_means = acc.mean(axis=0)
    return clean_mean - corr_means

def build_diff_table(configs):
    diffs      = {} 
    corruptions_ref = None

    for cfg_name, macs, mem, robust in configs:
        try:
            clean_accs, acc, corruptions = load_config(macs, mem, robust=robust)
            if corruptions_ref is None:
                corruptions_ref = corruptions
            diffs[cfg_name] = compute_diff(clean_accs, acc)
        except FileNotFoundError as e:
            print(f"  [AVISO] Arquivo não encontrado para {cfg_name}: {e}")
            diffs[cfg_name] = None

    C = len(corruptions_ref)

    data = {}
    for cfg_name, macs, mem, robust in configs:
        col_vals = []
        diff = diffs[cfg_name]

        if robust:
            trad_name = cfg_name.replace("_Robust", "")
            diff_trad = diffs.get(trad_name)
        else:
            diff_trad = None 

        for j in range(C):
            if diff is None:
                col_vals.append("N/A")
                continue

            val = diff[j]

            if robust and diff_trad is not None and diff_trad[j] is not None:
                prev = diff_trad[j]
                arrow = " ↓" if val < prev else " ↑"
            else:
                arrow = ""

            col_vals.append(f"{val:.2f}{arrow}")

        if diff is not None:
            mean_val = diff.mean()
            if robust and diff_trad is not None:
                prev_mean = diff_trad.mean()
                arrow = " ↓" if mean_val < prev_mean else " ↑"
            else:
                arrow = ""
            col_vals.append(f"{mean_val:.2f}{arrow}")
        else:
            col_vals.append("N/A")

        data[cfg_name] = col_vals

    index = corruptions_ref + ["Mean Diff"]
    return pd.DataFrame(data, index=index)

print("Gerando tabela diff 18M...")
df_18m = build_diff_table(CONFIGS_18M)
path_18m = os.path.join(OUT_DIR, "table_diff_18M.csv")
df_18m.to_csv(path_18m)
print(f"  Salvo: {path_18m}")
print(df_18m.to_string())

print("\nGerando tabela diff 52M...")
df_52m = build_diff_table(CONFIGS_52M)
path_52m = os.path.join(OUT_DIR, "table_diff_52M.csv")
df_52m.to_csv(path_52m)
print(f"  Salvo: {path_52m}")
print(df_52m.to_string())